# PCI DSS assistant

## Step 1 — PDF into a list of documents

In [1]:
# reload ingest.py / rag_helper.py / evaluation_utils.py automatically
# whenever they change on disk, without restarting the kernel
%load_ext autoreload
%autoreload 2

### Resuming after a restart

Run this one cell to rebuild everything that lives in memory. It is free and takes a
couple of seconds — nothing here calls the API.

**Do not use "Run All"**: the ground-truth cell in step 4 would regenerate the
questions, which costs money and makes the metrics from step 5 incomparable with
later steps.

In [2]:
import os

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

import ingest
from rag_helper import RAGBase

load_dotenv()

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

documents = ingest.load_documents()
index = ingest.build_index(documents)
pci_rag = RAGBase(index=index, llm_client=client)

ground_truth = pd.read_csv('data/ground_truth.csv').to_dict(orient='records')


def show(results):
    for r in results:
        print(f"page {r['page']:>3}  req {r['req_ids']}")


len(documents), len(ground_truth)

(261, 1305)

In [3]:
import ingest

In [4]:
# downloads the PCI DSS v4.0.1 PDF into data/ (~4.4 MB, skipped if already there)
ingest.download_pdf()

'data/pci-dss-v4_0_1.pdf'

In [5]:
documents = ingest.load_documents()
len(documents)

261

In [6]:
# what one document looks like
doc = documents[14]

doc['page'], doc['req_ids'], doc['requirement']

(59, '1.4.2', '1')

In [7]:
print(doc['text'])

Payment Card Industry Data Security Standard: Requirements and Testing Procedures, v4.0.1 
June 2024 
©2006 - 2024 PCI Security Standards Council, LLC. All Rights Reserved. 
Page 55 
 
Requirements and Testing Procedures 
Guidance 
Defined Approach Requirements 
Defined Approach Testing Procedures 
Purpose 
Ensuring that public access to a system 
component is specifically authorized reduces the 
risk of system components being unnecessarily 
exposed to untrusted networks. 
Good Practice 
System components that provide publicly 
accessible services, such as email, web, and 
DNS servers, are the most vulnerable to threats 
originating from untrusted networks.  
Ideally, such systems are placed within a 
dedicated trusted network that is public facing (for 
example, a DMZ) but that is separated via NSCs 
from more sensitive internal systems, which helps 
protect the rest of the network in the event these 
externally accessible systems are compromised. 
This functionality is intended to p

In [8]:
# how long are the documents?
lengths = [len(d['text']) for d in documents]

min(lengths), sum(lengths) // len(lengths), max(lengths)

(791, 2153, 3623)

## Step 2 — Text search

`minsearch` builds a TF-IDF index over the text fields and lets us filter on the
keyword fields. No server, no persistence: 261 documents are indexed in a moment.

In [9]:
index = ingest.build_index(documents)


def show(results):
    for r in results:
        print(f"page {r['page']:>3}  req {r['req_ids']}")

In [10]:
query = 'how long must audit logs be retained'
results = index.search(query, num_results=5)

show(results)

page 256  req 10.5, 10.5.1
page 134  req 5.3.4
page 243  req 10.2, 10.2.1, 10.2.1.1
page 244  req 10.2.1.2, 10.2.1.3, 10.2.1.4
page 248  req 10.3, 10.3.1


In [11]:
# the top hit, to check it really answers the question
print(results[0]['text'][:800])

Payment Card Industry Data Security Standard: Requirements and Testing Procedures, v4.0.1 
June 2024 
©2006 - 2024 PCI Security Standards Council, LLC. All Rights Reserved. 
Page 252 
 
Requirements and Testing Procedures 
Guidance 
10.5 Audit log history is retained and available for analysis. 
Defined Approach Requirements 
Defined Approach Testing Procedures 
Purpose 
Retaining historical audit logs for at least 12 
months is necessary because compromises often 
go unnoticed for significant lengths of time. 
Having centrally stored log history allows 
investigators to better determine the length of 
time a potential breach was occurring, and the 
possible system(s) impacted. By having three 
months of logs immediately available, an entity 
can quickly identify and minimize impact of a d


In [12]:
show(index.search('do we need MFA for remote access', num_results=5))

page 208  req 8.5, 8.5.1
page 206  req 8.4.3
page 202  req 8.4, 8.4.1
page 203  req 8.4.2
page 184  req 8.2.2, 8.2.3


### Requirement numbers, and why they nearly did not work

Half of the realistic questions about a standard are about a specific requirement
number. That turned out to be broken by default.

In [13]:
from minsearch import Index

# an index built exactly the way the course did it, with no extra parameters
plain = Index(text_fields=['text', 'req_ids'], keyword_fields=['requirement'])
plain.fit(documents)

plain.search('8.3.6', num_results=3)  # -> []

[]

Nothing. `minsearch` uses scikit-learn's `TfidfVectorizer`, whose default token
pattern is `\b\w\w+\b`: a token needs at least two word characters, and a dot is
not a word character. So `8.3.6` is split into `8`, `3`, `6`, each too short to
survive — requirement numbers never make it into the index.

Letting a dot stay inside a token fixes it. That is the one extra line in
`build_index`:

```python
vectorizer_params = {'token_pattern': r'(?u)\b\w[\w.]*\b'}
```

In [14]:
show(index.search('8.3.6', num_results=3))

page 194  req 8.3.6
page  69  req 2.2.2


In [15]:
# it helps ordinary questions too — 10.5.1 is the requirement about log retention
show(index.search('10.5.1', num_results=3))

page 256  req 10.5, 10.5.1
page 134  req 5.3.4


### Boosting and filtering

`boost_dict` weights the text fields against each other, `filter_dict` does exact
matching on a keyword field — the role `course` played in the course code.

In [16]:
# make an exact requirement number outweigh pages that merely mention it
show(index.search('8.3.6', num_results=3, boost_dict={'req_ids': 3.0, 'text': 1.0}))

page 194  req 8.3.6
page  69  req 2.2.2


In [17]:
# search only inside Requirement 3 (Protect Stored Account Data)
show(index.search(
    'can we store the card verification code',
    num_results=5,
    filter_dict={'requirement': '3'}
))

page  85  req 3.3.1.2
page  89  req 3.3.3
page  83  req 3.3, 3.3.1
page 113  req 3.7.9
page  84  req 3.3.1.1


Good enough to build on. Which boost values are actually best, and whether TF-IDF
beats embeddings here, is not something to decide by eye — that is measured in
step 5.

## Step 3 — First RAG

Search finds the pages; the LLM turns them into an answer. Three stages:
**R**etrieval, **A**ugmentation (putting what we found into the prompt),
**G**eneration.

In [18]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

In [19]:
from rag_helper import RAGBase

pci_rag = RAGBase(index=index, llm_client=client)

### Look at the pieces before running the whole thing

In [20]:
query = 'how long must we retain audit logs?'

search_results = pci_rag.search(query)
show(search_results)

page 244  req 10.2.1.2, 10.2.1.3, 10.2.1.4
page 243  req 10.2, 10.2.1, 10.2.1.1
page 256  req 10.5, 10.5.1
page 248  req 10.3, 10.3.1
page 246  req 10.2.1.5, 10.2.1.6


In [21]:
# what the model will actually see
prompt = pci_rag.build_prompt(query, search_results)

print(prompt[:1500])

QUESTION: how long must we retain audit logs?

CONTEXT:
[requirement 10.2.1.2, 10.2.1.3, 10.2.1.4, page 244]
Payment Card Industry Data Security Standard: Requirements and Testing Procedures, v4.0.1 
June 2024 
©2006 - 2024 PCI Security Standards Council, LLC. All Rights Reserved. 
Page 240 
 
Requirements and Testing Procedures 
Guidance 
Defined Approach Requirements 
Defined Approach Testing Procedures 
Purpose 
Accounts with increased access privileges, such 
as the “administrator” or “root” account, have the 
potential to significantly impact the security or 
operational functionality of a system. Without a 
log of the activities performed, an organization is 
cannot trace any issues resulting from an 
administrative mistake or misuse of privilege back 
to the specific action and account. 
Definitions 
The functions or activities considered to be 
administrative are beyond those performed by 
regular users as part of routine business 
functions. 
Refer to Appendix G for the defini

In [22]:
print(f'prompt length: {len(prompt)} characters')

prompt length: 10039 characters


### The whole pipeline

In [23]:
answer = pci_rag.rag(query)

print(answer)

Audit log history must be retained for at least 12 months, with at least the most recent three months immediately available for analysis. (req. 10.5.1, p. 252)


In [24]:
print(pci_rag.rag('do we need MFA for administrative access, or only for remote access?'))

You need MFA for **non-console administrative access** into the CDE, not only for remote access. Requirement 8.4.1 says MFA is implemented for **all non-console access into the CDE for personnel with administrative access**. (req. 8.4.1, p. 202)

MFA is also required for **all remote access originating from outside the entity’s network that could access or impact the CDE**. (req. 8.4.3, p. 202)

So, based on the standard here, it’s **both**:
- non-console administrative access into the CDE, and
- remote access outside the entity’s network that could access or impact the CDE. (req. 8.4.1, p. 202; req. 8.4.3, p. 202)




In [25]:
print(pci_rag.rag('can we store the CVV after the transaction is authorized?'))

No. The card verification code/CVV is sensitive authentication data, and it is not stored upon completion of the authorization process, even if encrypted. It must be rendered unrecoverable after authorization. (req. 3.3.1.2, p. 81; req. 3.3.1, p. 79)


In [26]:
print(pci_rag.rag('what exactly does requirement 8.3.6 ask for?'))

Requirement 8.3.6 says that if passwords/passphrases are used as an authentication factor to meet Requirement 8.3.1, they must meet both of these minimum complexity rules: a minimum length of 12 characters, or 8 characters if the system does not support 12, and they must contain both numeric and alphabetic characters. The testing procedure is to examine system configuration settings to verify the password/passphrase complexity parameters match all elements of the requirement. (req. 8.3.6, p. 194)

It also notes that this requirement is not intended to apply to user accounts on POS terminals that can access only one card number at a time, or to application/system accounts covered by section 8.6. (req. 8.3.6, p. 194)

It is a best practice until 31 March 2025, and after that it becomes required; until then, passwords must be at least seven characters per PCI DSS v3.2.1 Requirement 8.2.3. (req. 8.3.6, p. 194)


### Does it refuse when it should?

A compliance assistant that confidently answers questions the standard does not
cover is worse than one that stays silent. This question has nothing to do with
PCI DSS, so the answer should be "I don't know".

In [27]:
print(pci_rag.rag('what is the maximum fine under GDPR?'))

I don’t know. The provided PCI DSS context mentions GDPR only as an example of a legal reporting requirement, but it does not state the maximum GDPR fine. (req. 12.10.1, p. 332)


Working end to end. Two things we still cannot say anything about:

- how often search puts the right page in the top 5 — measured in step 5;
- whether this prompt is better than another one — measured in step 7.

Both need a set of questions with known answers, which is step 4.

## Step 4 — Ground truth

To measure search we need questions whose correct answer we already know.
Hand-labelling 261 pages is not realistic, so the LLM reads a page and invents
questions that page answers. The right answer is known by construction: it is
that page.

### Structured output

We do not want to parse a numbered list out of free text. `responses.parse` takes
a pydantic model and returns an object of that shape.

In [28]:
from pydantic import BaseModel


class Questions(BaseModel):
    questions: list[str]

### The generation prompt

One detail decides whether the whole evaluation is meaningful: **the questions must
not reuse the wording of the page**. If the LLM copies distinctive phrases,
TF-IDF finds the page trivially, hit rate comes out near 1.0, and we have measured
nothing. Real users ask in their own words, so the questions have to as well.

In [29]:
data_gen_instructions = '''
You emulate an engineer or an auditor preparing for a PCI DSS assessment.

You are given one page of the PCI DSS v4.0.1 standard. Formulate 5 questions this
person might ask that this page answers. The questions must be answerable from the
page alone, complete, and specific.

Use as few words from the page as possible — ask in your own words, the way people
phrase questions at work. Vary them: some about what is required, some about how it
is verified, some about a specific requirement number.

Do not mention "this page" or "the context" in the questions.
'''.strip()

### One document first

In [30]:
import json

doc = documents[14]
user_prompt = json.dumps(doc)

print(doc['page'], doc['req_ids'])

59 1.4.2


In [31]:
from evaluation_utils import llm_structured

result, usage = llm_structured(client, data_gen_instructions, user_prompt, Questions)

result.questions

['What inbound traffic is allowed from an untrusted network into a trusted one under 1.4.2?',
 'How do you test compliance with 1.4.2 for network security controls?',
 'What should happen to inbound traffic that is neither a public-service connection nor a stateful reply?',
 'Does 1.4.2 allow UDP or other connectionless protocols?',
 'Which PCI DSS requirement covers restricting inbound traffic from untrusted to trusted networks?']

In [32]:
from evaluation_utils import calc_price

usage.input_tokens, usage.output_tokens, calc_price(usage)

(850,
 104,
 {'input_cost': 0.0006374999999999999,
  'output_cost': 0.000468,
  'total_cost': 0.0011055})

### One function per document

`document` holds the page number: that is the label we will check retrieval against.

In [33]:
from evaluation_utils import llm_structured_retry


def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    records = []

    for q in out.questions:
        records.append({
            'question': q,
            'document': doc['page']
        })

    return records, usage

In [34]:
records, usage = generate_ground_truth(documents[0])

records

[{'question': 'What does 1.1.1 require for the policies and procedures tied to Requirement 1?',
  'document': 44},
 {'question': 'How do you prove those Requirement 1 policies and procedures are being managed properly?',
  'document': 44},
 {'question': 'Are the network security control procedures supposed to be updated only on a regular schedule, or sooner after changes happen?',
  'document': 44},
 {'question': 'What does Requirement 1.1 say about how the process for installing and maintaining network security controls should be handled?',
  'document': 44},
 {'question': 'Which people need to know the Requirement 1 policies and operational procedures under 1.1.1?',
  'document': 44}]

### All 261 pages

Sequentially this would take a while, so we run several requests at once.
`map_progress` keeps the order and shows a progress bar.

In [39]:
from concurrent.futures import ThreadPoolExecutor

from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/261 [00:00<?, ?it/s]

In [40]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

1305

In [41]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.2670757500000001

### Save it

Generating this again costs money and gives different questions, which would make
metrics from different steps incomparable. Generate once, commit the file, reuse it
in steps 5, 6 and 7.

In [42]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth.to_csv('data/ground_truth.csv', index=False)

df_ground_truth.head()

,question,document
0,What has to be true for the security policies ...,44
1,How do you check whether Requirement 1 policie...,44
2,What does PCI mean by the processes for instal...,44
3,Does 1.1.1 require policies and procedures to ...,44
4,What should the assessor look at or ask to ver...,44


### Sanity check

Read a few questions and ask yourself: could someone answer this without the page in
front of them? Does it copy the page's wording? Bad questions here quietly corrupt
every number in the next three steps.

In [43]:
df_ground_truth.sample(10, random_state=1)

,question,document
644,"Under Requirement 8.4.3, can remote access be ...",206
1032,Which tasks are covered by the quarterly revie...,306
163,How is compliance with 3.2.1 verified during a...,81
1009,"For requirement 12.3, what is the purpose of d...",300
909,What should the targeted risk analysis for 11....,272
268,What happens if a cleartext key component hold...,109
1134,Does the incident response plan need to be tes...,332
983,How do you verify compliance with requirement ...,292
1113,Does 12.8.5 cover requirements owned by the en...,325
904,What evidence is reviewed to prove internal re...,270


## Step 5 — Search evaluation

For every ground-truth question we run search and mark where the correct page
landed in the results:

```
[1, 0, 0, 0, 0]   correct page came first
[0, 0, 1, 0, 0]   third
[0, 0, 0, 0, 0]   not found at all
```

**Hit rate** — the share of questions where the correct page appears anywhere in the
results. It answers *did we even give the model a chance*: if the page is not in the
context, no prompt can save the answer.

**MRR** also accounts for position — rank 1 scores 1.0, rank 2 scores 0.5, rank 3
scores 0.33. It answers *how high*, which matters because context is finite: a page
that is reliably fifth disappears the moment you shrink the context to three.

In [44]:
import pandas as pd

df_ground_truth = pd.read_csv('data/ground_truth.csv')
ground_truth = df_ground_truth.to_dict(orient='records')

len(ground_truth), ground_truth[0]

(1305,
 {'question': 'What has to be true for the security policies and procedures in Requirement 1 to be compliant?',
  'document': 44})

### Relevance for one question

In [45]:
def text_search(query, num_results=5):
    boost_dict = {'req_ids': 3.0, 'text': 1.0}

    return index.search(query, num_results=num_results, boost_dict=boost_dict)

In [46]:
q = ground_truth[0]
results = text_search(q['question'])

print(q['question'])
print('correct page:', q['document'])

for d in results:
    print(f"  page {d['page']}  ->  {d['page'] == q['document']}")

What has to be true for the security policies and procedures in Requirement 1 to be compliant?
correct page: 44
  page 44  ->  True
  page 66  ->  False
  page 166  ->  False
  page 241  ->  False
  page 215  ->  False


In [47]:
import importlib
import evaluation_utils

importlib.reload(evaluation_utils)

from evaluation_utils import compute_relevance, evaluate, hit_rate, mrr

In [48]:
from evaluation_utils import compute_relevance

compute_relevance(q, text_search)

[1, 0, 0, 0, 0]

### The metrics themselves

Both are a few lines. Worth reading them rather than importing blindly.

In [49]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
]

# hit rate: 3 of 4 questions found the page  -> 0.75
# mrr:      (1 + 0.5 + 0 + 0.333) / 4        -> 0.458

from evaluation_utils import hit_rate, mrr

hit_rate(example), mrr(example)

(0.75, 0.4583333333333333)

### Evaluating the whole ground truth

In [50]:
from evaluation_utils import evaluate

evaluate(ground_truth, text_search)

  0%|          | 0/1305 [00:00<?, ?it/s]

{'hit_rate': 0.9578544061302682, 'mrr': 0.8903959131545344}

### Comparing configurations

The boost of 3.0 on `req_ids` was a guess. Now we can check it instead of believing
it. Same questions, same metrics, different settings.

In [51]:
def make_search(boost_dict, num_results=5):
    def search_function(query):
        return index.search(query, num_results=num_results, boost_dict=boost_dict)

    return search_function


configurations = {
    'no boost':        make_search(None),
    'req_ids x 0.5':   make_search({'req_ids': 0.5, 'text': 1.0}),
    'req_ids x 3':     make_search({'req_ids': 3.0, 'text': 1.0}),
    'req_ids x 10':    make_search({'req_ids': 10.0, 'text': 1.0}),
    'text only':       make_search({'req_ids': 0.0, 'text': 1.0}),
}

In [52]:
rows = []

for name, search_function in configurations.items():
    print(name)
    metrics = evaluate(ground_truth, search_function)
    rows.append({'configuration': name, **metrics})

df_results = pd.DataFrame(rows).sort_values('mrr', ascending=False)
df_results

no boost


  0%|          | 0/1305 [00:00<?, ?it/s]

req_ids x 0.5


  0%|          | 0/1305 [00:00<?, ?it/s]

req_ids x 3


...

req_ids x 10


  0%|          | 0/1305 [00:00<?, ?it/s]

text only


  0%|          | 0/1305 [00:00<?, ?it/s]

,configuration,hit_rate,mrr
0,no boost,0.957854,0.890396
2,req_ids x 3,0.957854,0.890396
3,req_ids x 10,0.957854,0.890013
1,req_ids x 0.5,0.957854,0.889630
4,text only,0.862835,0.717803


### How many results to retrieve

In [53]:
rows = []

for n in [3, 5, 10]:
    print(f'num_results={n}')
    metrics = evaluate(ground_truth, make_search({'req_ids': 3.0, 'text': 1.0}, num_results=n))
    rows.append({'num_results': n, **metrics})

pd.DataFrame(rows)

num_results=3


  0%|          | 0/1305 [00:00<?, ?it/s]

num_results=5


  0%|          | 0/1305 [00:00<?, ?it/s]

num_results=10


  0%|          | 0/1305 [00:00<?, ?it/s]

,num_results,hit_rate,mrr
0,3,0.935632,0.885185
1,5,0.957854,0.890396
2,10,0.976245,0.892953


Retrieving more can only raise hit rate, so the number alone is not a reason to
increase it: every extra page costs tokens and dilutes the context. MRR barely moves
with `num_results`, which is the point — it measures ranking, not recall.

### Save the results

These numbers are the baseline. Step 6 has to beat them.

In [54]:
df_results.to_csv('data/eval_text_search.csv', index=False)

df_results

,configuration,hit_rate,mrr
0,no boost,0.957854,0.890396
2,req_ids x 3,0.957854,0.890396
3,req_ids x 10,0.957854,0.890013
1,req_ids x 0.5,0.957854,0.889630
4,text only,0.862835,0.717803


### Why the boost made no difference

`no boost`, `x3` and `x10` came out identical to six decimals, yet dropping `req_ids`
entirely costs ~8 points of hit rate. So the field matters enormously and its weight
does not.

The likely reason: some generated questions quote a requirement number literally
("what does 8.3.6 require?"). A number match is unique, so those questions are found
no matter how the field is weighted — and they inflate the overall score. Real users
mostly ask without a number, so the two groups need separate numbers.

In [55]:
import re

HAS_NUMBER = re.compile(r'\b(?:A\d\.)?\d{1,2}\.\d{1,2}(?:\.\d{1,2})*\b')

with_number = [q for q in ground_truth if HAS_NUMBER.search(q['question'])]
without_number = [q for q in ground_truth if not HAS_NUMBER.search(q['question'])]

len(with_number), len(without_number)

(788, 517)

In [56]:
for q in with_number[:3]:
    print('WITH   ', q['question'])
for q in without_number[:3]:
    print('WITHOUT', q['question'])

WITH    What does PCI mean by the processes for installing and maintaining network security controls in Requirement 1.1?
WITH    Does 1.1.1 require policies and procedures to be documented, current, used, and known by impacted staff?
WITH    What should the assessor look at or ask to verify 1.1.1?
WITHOUT What has to be true for the security policies and procedures in Requirement 1 to be compliant?
WITHOUT How do you check whether Requirement 1 policies and procedures are being managed correctly?
WITHOUT What has to be documented and assigned for Requirement 1 activities, and how do you show staff understand it?


In [57]:
rows = []

for group_name, group in [('with number', with_number), ('without number', without_number)]:
    for config_name in ['no boost', 'req_ids x 3', 'text only']:
        print(group_name, '|', config_name)
        metrics = evaluate(group, configurations[config_name])
        rows.append({'group': group_name, 'configuration': config_name, **metrics})

df_split = pd.DataFrame(rows)
df_split

with number | no boost


  0%|          | 0/788 [00:00<?, ?it/s]

with number | req_ids x 3


  0%|          | 0/788 [00:00<?, ?it/s]

with number | text only


...

without number | req_ids x 3


  0%|          | 0/517 [00:00<?, ?it/s]

without number | text only


  0%|          | 0/517 [00:00<?, ?it/s]

,group,configuration,hit_rate,mrr
0,with number,no boost,0.977157,0.957614
1,with number,req_ids x 3,0.977157,0.957614
2,with number,text only,0.822335,0.674958
3,without number,no boost,0.928433,0.787943
4,without number,req_ids x 3,0.928433,0.787943
5,without number,text only,0.924565,0.783108


The number in the README should be the "without number" one: that is the honest
measure of how the assistant behaves for a person who does not already know which
requirement to look at.

In [58]:
df_split.to_csv('data/eval_text_search_split.csv', index=False)

## Step 6 — Vector and hybrid search

TF-IDF matches words. Embeddings match meaning: "how long do we keep logs" and
"audit log history is retained for twelve months" share almost no words but sit close
together in vector space.

The bar is high — text search already reaches hit rate 0.942 / MRR 0.809 on
plain-language questions.

In [59]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

model.max_seq_length

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

256

### A problem worth noticing before we start

`max_seq_length` is **256 tokens**, and our pages are around 500. Everything past the
halfway point of a page is silently thrown away before it is ever embedded.

Nothing warns you about this. Let's see how much text we are actually losing.

In [61]:
tokenizer = model.tokenizer

token_counts = [len(tokenizer.encode(d['text'])) for d in documents]

pd.Series(token_counts).describe()

count    261.000000
mean     394.164751
std      106.268599
min      147.000000
25%      318.000000
50%      383.000000
75%      473.000000
max      687.000000
dtype: float64

In [62]:
truncated = sum(1 for n in token_counts if n > model.max_seq_length)

f'{truncated} of {len(documents)} pages get truncated'

'244 of 261 pages get truncated'

### Embedding the documents

In [63]:
from tqdm.auto import tqdm

texts = [d['text'] for d in documents]

vectors = model.encode(texts, batch_size=32, show_progress_bar=True)

vectors.shape

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

(261, 384)

In [64]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(vectors, documents)

In [65]:
query = 'how long must we retain audit logs?'
query_vector = model.encode(query)

show(vindex.search(query_vector, num_results=5))

page 256  req 10.5, 10.5.1
page 250  req 10.3.4
page 252  req 10.4.1.1
page 243  req 10.2, 10.2.1, 10.2.1.1
page 248  req 10.3, 10.3.1


### Measuring it

Same ground truth, same metrics as step 5 — that is the whole point of having saved
them.

In [66]:
from evaluation_utils import evaluate


def make_vector_search(embedder, vector_index, num_results=5):
    def search_function(query, num_results=num_results):
        return vector_index.search(embedder.encode(query), num_results=num_results)

    return search_function


vector_search = make_vector_search(model, vindex)

evaluate(ground_truth, vector_search)

  0%|          | 0/1305 [00:00<?, ?it/s]

{'hit_rate': 0.671264367816092, 'mrr': 0.5079182630906771}

### A second embedding model

`all-MiniLM-L6-v2` is a general-purpose model with a 256-token window.
`multi-qa-MiniLM-L6-cos-v1` is the same size but trained specifically for
question-to-passage retrieval and takes 512 tokens, so it sees a whole page.

In [67]:
model_qa = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')

model_qa.max_seq_length

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

512

In [68]:
vectors_qa = model_qa.encode(texts, batch_size=32, show_progress_bar=True)

vindex_qa = VectorSearch()
vindex_qa.fit(vectors_qa, documents)

vector_search_qa = make_vector_search(model_qa, vindex_qa)

evaluate(ground_truth, vector_search_qa)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/1305 [00:00<?, ?it/s]

{'hit_rate': 0.6789272030651341, 'mrr': 0.5221711366538958}

### Hybrid search

Two ranked lists have to be merged, and their scores cannot simply be added: TF-IDF
similarity and embedding cosine live on different scales, so "0.7" means different
things in each. Reciprocal Rank Fusion combines **positions** instead:

```
score(doc) = sum over lists of  1 / (k + rank)
```

A page ranked first by both retrievers scores `1/61 + 1/61`. One found by only a
single retriever scores `1/61`. `k = 60` is conventional; it flattens the gap between
the top few ranks, so a page has to do well in both lists to win.

The implementation is in `rag_helper.reciprocal_rank_fusion` — worth reading, it is
twelve lines.

In [69]:
from rag_helper import reciprocal_rank_fusion

# what fusion does, on made-up input
list_a = [{'page': 1}, {'page': 2}, {'page': 3}]
list_b = [{'page': 3}, {'page': 1}, {'page': 9}]

[d['page'] for d in reciprocal_rank_fusion([list_a, list_b], num_results=4)]

[1, 3, 2, 9]

Page 1 wins (first and second), page 3 is close behind (third and first), then the
pages each retriever found alone.

In [70]:
def make_hybrid_search(embedder, vector_index, candidates=10, num_results=5):
    def search_function(query, num_results=num_results):
        text_results = index.search(query, num_results=candidates)
        vector_results = vector_index.search(embedder.encode(query), num_results=candidates)

        return reciprocal_rank_fusion(
            [text_results, vector_results],
            num_results=num_results
        )

    return search_function


hybrid_search = make_hybrid_search(model_qa, vindex_qa)

evaluate(ground_truth, hybrid_search)

  0%|          | 0/1305 [00:00<?, ?it/s]

{'hit_rate': 0.9509578544061302, 'mrr': 0.7479438058748428}

### Everything side by side

Including the split by question type, because that is where the honest number lives.

In [ ]:
approaches = {
    'text (TF-IDF)': text_search,
    'vector (MiniLM)': vector_search,
    'vector (multi-qa)': vector_search_qa,
    'hybrid (RRF)': hybrid_search,
}

rows = []

for name, search_function in approaches.items():
    print(name)
    overall = evaluate(ground_truth, search_function)
    plain = evaluate(without_number, search_function)
    rows.append({
        'approach': name,
        'hit_rate': overall['hit_rate'],
        'mrr': overall['mrr'],
        'hit_rate_plain': plain['hit_rate'],
        'mrr_plain': plain['mrr'],
    })

df_retrieval = pd.DataFrame(rows).sort_values('mrr_plain', ascending=False)
df_retrieval

text (TF-IDF)


  0%|          | 0/1305 [00:00<?, ?it/s]

  0%|          | 0/517 [00:00<?, ?it/s]

vector (MiniLM)


  0%|          | 0/1305 [00:00<?, ?it/s]

  0%|          | 0/517 [00:00<?, ?it/s]

vector (multi-qa)


  0%|          | 0/1305 [00:00<?, ?it/s]

### Routing: the two question types want different retrievers

The table says something more specific than "hybrid is best":

| | with a number | plain language |
|---|---|---|
| text (TF-IDF) | excellent — exact match on a unique token | 0.788 MRR |
| vector | nearly useless — `8.3.6` carries no meaning to an embedding model | 0.702 MRR |
| hybrid | *worse* than text — fusion drags the exact match down | **0.803 MRR** |

On a question quoting a number, TF-IDF puts the right page first and fusion then mixes
in a vector ranking where that page sits far down, pushing it off the top. Merging
actively hurts.

So: detect a requirement number in the question, and if there is one, skip fusion.

In [ ]:
from rag_helper import REQ_NUMBER_RE


def make_routed_search(embedder, vector_index, candidates=10, num_results=5):
    hybrid = make_hybrid_search(embedder, vector_index, candidates, num_results)

    def search_function(query, num_results=num_results):
        if REQ_NUMBER_RE.search(query):
            return index.search(query, num_results=num_results)

        return hybrid(query, num_results=num_results)

    return search_function


routed_search = make_routed_search(model_qa, vindex_qa)

In [ ]:
rows = []

for name, search_function in {**approaches, 'routed (text | hybrid)': routed_search}.items():
    print(name)
    overall = evaluate(ground_truth, search_function)
    plain = evaluate(without_number, search_function)
    numbered = evaluate(with_number, search_function)
    rows.append({
        'approach': name,
        'mrr': overall['mrr'],
        'hit_rate': overall['hit_rate'],
        'mrr_plain': plain['mrr'],
        'mrr_numbered': numbered['mrr'],
    })

df_retrieval = pd.DataFrame(rows).sort_values('mrr', ascending=False)
df_retrieval

In [ ]:
df_retrieval.to_csv('data/eval_retrieval.csv', index=False)

df_retrieval

Routing should take the best column from each row: text-search numbers on numbered
questions, hybrid numbers on plain ones. That is the configuration the application
uses — `RAGHybrid(route_numbers=True)` in `rag_helper.py`.

In [ ]:
from rag_helper import RAGHybrid

assistant = RAGHybrid(
    text_index=index,
    vector_index=vindex_qa,
    embedder=model_qa,
    llm_client=client,
)

print(assistant.rag('how long must we retain audit logs?'))

In [ ]:
print(assistant.rag('what exactly does requirement 8.3.6 ask for?'))

### How many candidates should fusion see?

`candidates` is how many results each retriever contributes before merging. Too few
and fusion has nothing to reorder; too many and noise from the weaker retriever
creeps up the list.

In [ ]:
rows = []

for c in [5, 10, 20, 30]:
    print(f'candidates={c}')
    metrics = evaluate(without_number, make_hybrid_search(model_qa, vindex_qa, candidates=c))
    rows.append({'candidates': c, **metrics})

pd.DataFrame(rows)